# Structured Output with Pydantic and LangGraph
## Type-Safe LLM Outputs with Pydantic
⏱ ~10 min

**Structured output** is one of the highest-leverage patterns in applied LLM engineering. Instead of parsing free-form text, you define an exact Python schema and the model fills it in — validated automatically. By the end of this workshop you will understand *why* structured output matters, *how* Pydantic and `with_structured_output()` work together, and *how* to wire them into a LangGraph pipeline that searches the web and extracts clean, typed data.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why structured output? The parsing problem |
| 2 | **Pydantic Schemas** — Defining output shapes with `BaseModel` |
| 3 | **`with_structured_output()`** — Binding a schema to an LLM |
| 4 | **Field Descriptions** — How metadata guides the model |
| 5 | **LangGraph Pipeline** — Two-node search-then-extract graph |
| 6 | **Schema Variants** — Nested models, Literal types, optional fields |
| 7 | **Error Handling** — Validation failures and retry strategies |
| ★ | **Advanced Techniques** (bonus) |

---

### Prerequisites
- Python 3.10+ (a `.venv` with the requirements already installed)
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)
- No additional API keys required — web search uses DuckDuckGo (free, no key)

### Key References
> Brown, T., Mann, B., et al. (2020). *Language Models are Few-Shot Learners.* NeurIPS 2020. https://arxiv.org/abs/2005.14165  
> Wei, J., et al. (2022). *Emergent Abilities of Large Language Models.* TMLR. https://arxiv.org/abs/2206.07682  
> OpenAI. (2023). *Function Calling and Structured Outputs.* https://platform.openai.com/docs/guides/structured-outputs  
> Pydantic documentation: https://docs.pydantic.dev  
> LangChain structured output guide: https://python.langchain.com/docs/concepts/structured_outputs/

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "langchain",
            "langchain-openai",
            "langchain-community",
            "langgraph",
            "pydantic",
            "duckduckgo-search",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import json
import os
from typing import Literal, Optional, TypedDict

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

# ── Core imports ──────────────────────────────────────────────────────────────
from duckduckgo_search import DDGS
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field, ValidationError

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
key_ok = bool(key) and key.startswith("sk-")
print(f"OPENAI_API_KEY present and valid-looking: {key_ok}")
if not key_ok:
    print()
    print("  ACTION REQUIRED — add your key before running LLM cells.")
    print("  Local: echo 'OPENAI_API_KEY=sk-...' >> .env")
    print("  Colab: Secrets panel → Add secret 'OPENAI_API_KEY'")

---

## Part 1 — What Is Structured Output and Why Does It Exist?

---

### The Parsing Problem

LLMs return free-form text by default. When your downstream code needs structured data — a dict, a typed object, a validated record — you have to parse that text. This is brittle:

```
# What you asked for:
"Return a JSON object with keys: name, founded_year, headquarters"

# What the model returns (actual examples):
"Here is the JSON you requested:\n{...}"     <- wrapped in prose
"```json\n{...}\n```"                         <- wrapped in markdown
{"Name": ..., "foundedYear": ..., "HQ": ...} <- wrong key names
{"name": ..., "founded_year": "2015"}        <- year as string, not int
```

Every one of these requires custom string-munging code that breaks on the next model update.

---

### Three Approaches Compared

| Approach | How it works | Reliability | Effort |
|----------|-------------|-------------|--------|
| **Prompt engineering** | "Return only JSON with keys X, Y, Z" | Low — model ignores instructions | Low |
| **Regex / string parsing** | Extract JSON from the response text | Medium — brittle on format changes | Medium |
| **`with_structured_output()`** | Model constrained to emit valid schema | High — enforced at generation | Low |

Modern OpenAI models support **JSON mode** and **function calling**, which allow the model to be constrained at the token generation level — it *cannot* emit text that violates the schema. `with_structured_output()` is LangChain's cross-model wrapper for this feature.

---

### Key Vocabulary

| Term | Definition |
|------|------------|
| **Pydantic `BaseModel`** | A Python class whose fields are type-checked at runtime |
| **`Field(description=...)`** | Metadata attached to a field — the LLM reads this to know what to fill in |
| **`with_structured_output()`** | LangChain method that binds a schema to an LLM and returns a typed object |
| **JSON Schema** | The wire format that OpenAI's API actually receives — Pydantic generates it |
| **Function calling** | The OpenAI API mechanism that forces structured output |
| **`ValidationError`** | Pydantic exception raised when the model output does not match the schema |
| **`model_dump()`** | Pydantic method to convert a model instance to a plain dict |

### Structured Output Architecture

```
YOUR CODE                        LANGCHAIN / OPENAI
──────────────────────────────   ──────────────────────────────────────

  class MyModel(BaseModel):        Pydantic generates JSON Schema
      name: str                    {
      year: int          ──────►     "type": "object",
      tags: list[str]               "properties": {
                                      "name":  {"type": "string"},
                                      "year":  {"type": "integer"},
                                      "tags":  {"type": "array", ...}
                                    },
                                    "required": ["name", "year", "tags"]
                                  }
                                        │
                                        ▼
  llm.with_structured_output(     OpenAI function-calling API
      MyModel                     (schema sent as "tools" parameter)
  )                                     │
                                        ▼
  structured_llm.invoke(          Model constrained to emit
      [SystemMessage(...),         ONLY valid schema tokens
       HumanMessage(...)]          — no prose wrapping possible
  )                                     │
                                        ▼
  result: MyModel  ◄──────────── Pydantic validates + constructs
                                   typed Python object
  result.name   # str
  result.year   # int              Runtime type guarantee
  result.tags   # list[str]
```

> **Key insight:** The schema travels to the model as structured metadata (via the function-calling API), not as natural language instructions in the prompt. This is why `with_structured_output()` is far more reliable than asking the model to "return JSON".

In [ ]:
# ─── 1-A: The parsing problem in action ──────────────────────────────────────
# Show what happens when you ask for JSON without structured output.

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = (
    "Tell me about Python (the programming language). "
    "Return a JSON object with keys: name, created_year, creator, summary."
)

raw_response = llm.invoke(prompt)
print("Raw model output (type:", type(raw_response.content).__name__, "):")
print(repr(raw_response.content[:300]))
print()
print("Problems with this approach:")
print("  - Output may be wrapped in markdown code fences")
print("  - Key names may differ from what you specified")
print("  - Types are not guaranteed (year might come back as a string)")
print("  - Any of this breaks json.loads() silently or loudly")

---

## Part 2 — Defining Schemas with Pydantic

---

### Why Pydantic?

Pydantic is Python's most widely used data validation library. It gives you:
- **Runtime type enforcement** — fields are coerced and validated on creation
- **Automatic JSON Schema generation** — OpenAI's API needs JSON Schema, not Python classes
- **Descriptive field metadata** — `Field(description=...)` is read by the LLM
- **Serialization** — `model_dump()` and `model_dump_json()` for free

---

### Pydantic vs Alternatives

| Schema approach | Pydantic `BaseModel` | `TypedDict` | Raw JSON Schema |
|----------------|----------------------|-------------|-----------------|
| Runtime validation | Yes — raises `ValidationError` | No — type hint only | No |
| Field descriptions | `Field(description=...)` | Not supported | Manual |
| Auto JSON Schema | Yes — `model.model_json_schema()` | Partial | Manual |
| LangChain support | Full | Full | Full |
| Best for | Production pipelines | Simple typing | Custom schemas |

> **Rule of thumb:** Use `BaseModel` when you want validation. Use `TypedDict` when you only need type hints for IDE support. Use raw JSON Schema only when integrating with non-LangChain systems.

In [ ]:
# ─── 2-A: Define a Pydantic model and inspect its JSON Schema ─────────────────

class LanguageProfile(BaseModel):
    name: str = Field(description="The programming language name")
    created_year: int = Field(description="Year the language was first released")
    creator: str = Field(description="Name of the primary creator or organization")
    paradigms: list[str] = Field(
        description="Programming paradigms, e.g. ['functional', 'object-oriented']"
    )
    summary: str = Field(
        description="2-3 sentence description of the language's key characteristics"
    )


# This is what LangChain sends to the OpenAI API
schema = LanguageProfile.model_json_schema()
print("JSON Schema sent to the API:")
print(json.dumps(schema, indent=2))

In [ ]:
# ─── 2-B: Pydantic validation in action ──────────────────────────────────────
# Shows what happens when data does and does not match the schema.

# Valid construction
valid = LanguageProfile(
    name="Python",
    created_year=1991,
    creator="Guido van Rossum",
    paradigms=["object-oriented", "functional", "procedural"],
    summary="Python is a high-level, interpreted language emphasizing readability.",
)
print("Valid model:")
print(f"  name: {valid.name!r} ({type(valid.name).__name__})")
print(f"  created_year: {valid.created_year!r} ({type(valid.created_year).__name__})")
print(f"  paradigms: {valid.paradigms}")
print()

# Pydantic coerces types where possible
coerced = LanguageProfile(
    name="Rust",
    created_year="2015",  # string -> int coercion
    creator="Graydon Hoare",
    paradigms=["systems", "functional"],
    summary="Rust guarantees memory safety without a garbage collector.",
)
print("Coerced model (year was passed as string '2015'):")
print(f"  created_year: {coerced.created_year!r} ({type(coerced.created_year).__name__})")
print()

# Validation failure
try:
    bad = LanguageProfile(
        name="Go",
        created_year="not-a-year",  # cannot coerce to int
        creator="Robert Griesemer",
        paradigms=[],
        summary="Go is a compiled, statically typed language.",
    )
except ValidationError as e:
    print("ValidationError caught:")
    for err in e.errors():
        print(f"  field={err['loc']}  type={err['type']}  msg={err['msg']}")

In [ ]:
# ─── 2-C: Serialization — model_dump() and model_dump_json() ─────────────────
# After extraction, convert the Pydantic object to a dict or JSON
# for storage, logging, or passing to the next pipeline stage.

print("model_dump() -> plain dict:")
print(json.dumps(valid.model_dump(), indent=2))
print()
print("model_dump_json() -> JSON string (compact):")
print(valid.model_dump_json())

### Exercise 1 — Design a Schema

Design a Pydantic model called `BookProfile` for extracting information about books.

**Required fields:**
- `title` (str)
- `author` (str)
- `published_year` (int)
- `genres` (list of strings)
- `one_line_summary` (str — exactly one sentence)

**Goal:** Write clear `Field(description=...)` for every field. Then call `model_json_schema()` and check that the descriptions appear in the output.

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────

class BookProfile(BaseModel):
    title: str = Field(description="TODO: describe this field")
    author: str = Field(description="TODO: describe this field")
    published_year: int = Field(description="TODO: describe this field")
    genres: list[str] = Field(description="TODO: describe this field")
    one_line_summary: str = Field(description="TODO: describe this field")


# TODO: print the JSON schema and verify your descriptions appear
# TODO: instantiate one valid BookProfile to confirm the model validates correctly

---

## Part 3 — `with_structured_output()`: Binding a Schema to an LLM

---

### How It Works

`llm.with_structured_output(MyModel)` returns a new runnable that:
1. Converts `MyModel` to JSON Schema
2. Passes the schema to the model via function calling (not in the prompt text)
3. Instructs the model to call that "function" with its answer
4. Parses the returned arguments through Pydantic
5. Returns a fully validated `MyModel` instance — not a string, not a dict

---

### Output Parsing Methods

| Method | Returns | When to use |
|--------|---------|-------------|
| `with_structured_output(MyModel)` | `MyModel` instance | Default — full Pydantic validation |
| `with_structured_output(MyModel, include_raw=True)` | `{"raw": AIMessage, "parsed": MyModel, "parsing_error": ...}` | When you need the raw response for debugging |
| `with_structured_output({"type": "object", ...})` | `dict` | When you have a JSON Schema but no Pydantic model |
| `JsonOutputParser()` | `dict` | Lightweight — parses JSON from any output, no schema |
| `PydanticOutputParser(pydantic_object=MyModel)` | `MyModel` | Legacy; `with_structured_output` is preferred |

In [ ]:
# ─── 3-A: Basic with_structured_output() call ─────────────────────────────────

structured_llm = llm.with_structured_output(LanguageProfile)

result = structured_llm.invoke(
    "Tell me about the Python programming language."
)

print(f"Return type: {type(result).__name__}")
print(f"name        : {result.name}")
print(f"created_year: {result.created_year}  (type: {type(result.created_year).__name__})")
print(f"creator     : {result.creator}")
print(f"paradigms   : {result.paradigms}")
print(f"summary     : {result.summary[:100]}...")

In [ ]:
# ─── 3-B: include_raw=True — inspect the raw API response ─────────────────────
# Useful for debugging: see exactly what the model returned before parsing.

structured_llm_raw = llm.with_structured_output(LanguageProfile, include_raw=True)

raw_result = structured_llm_raw.invoke("Tell me about Rust.")

print("Keys in raw result:", list(raw_result.keys()))
print()
print("parsed (Pydantic object):")
print(f"  type: {type(raw_result['parsed']).__name__}")
print(f"  name: {raw_result['parsed'].name}")
print()
print("parsing_error:", raw_result.get("parsing_error"))
print()
print("raw (AIMessage tool_calls):")
if raw_result["raw"].tool_calls:
    tc = raw_result["raw"].tool_calls[0]
    print(f"  function name : {tc['name']}")
    print(f"  args (truncated): {str(tc['args'])[:200]}")

In [ ]:
# ─── 3-C: Batch extraction — profile multiple subjects ────────────────────────
# with_structured_output works with .batch() just like any other runnable.

languages = ["Go", "TypeScript", "Haskell"]

results = structured_llm.batch(
    [f"Tell me about the {lang} programming language." for lang in languages]
)

print(f"Extracted {len(results)} profiles:\n")
for profile in results:
    print(f"  {profile.name} ({profile.created_year}) — by {profile.creator}")
    print(f"  Paradigms: {', '.join(profile.paradigms)}")
    print()

---

## Part 4 — Field Descriptions: How Metadata Guides the Model

---

### Why Field Descriptions Matter

The `Field(description=...)` string is included in the JSON Schema that the model receives. It acts like an inline instruction for each field. **The model reads the description, not just the field name.**

```python
# Vague — model guesses what "confidence" means
confidence: str = Field(description="confidence")

# Precise — model knows exactly what to put here
confidence: str = Field(
    description="Reliability estimate: 'high' if multiple sources agree, "
                "'medium' if one source, 'low' if speculative or conflicting"
)
```

Good descriptions reduce hallucination, improve consistency, and remove the need for long system prompt instructions.

---

### Schema Design Checklist

- Describe **what** the field should contain, not just **what it is**
- For constrained values, list the allowed options in the description (or use `Literal[...]`)
- For list fields, specify expected length (e.g., "3-5 items")
- For text fields, specify expected length (e.g., "one sentence", "2-3 sentences")
- Use `Optional[T]` and `default=None` when a field may not exist in the source material

In [ ]:
# ─── 4-A: Compare vague vs precise descriptions ───────────────────────────────

class VagueProfile(BaseModel):
    name: str = Field(description="name")
    description: str = Field(description="description")
    score: str = Field(description="score")


class PreciseProfile(BaseModel):
    name: str = Field(description="Full official name of the entity")
    description: str = Field(
        description="Exactly two sentences: the first states what it is, "
                    "the second states why it is significant"
    )
    score: str = Field(
        description="Adoption score: 'high' if used by millions of developers, "
                    "'medium' if niche but established, 'low' if experimental"
    )


topic = "Kubernetes"

print("=== Vague descriptions ===")
vague_llm = llm.with_structured_output(VagueProfile)
v = vague_llm.invoke(f"Tell me about {topic}.")
print(f"  name       : {v.name}")
print(f"  description: {v.description[:120]}")
print(f"  score      : {v.score}")

print()
print("=== Precise descriptions ===")
precise_llm = llm.with_structured_output(PreciseProfile)
p = precise_llm.invoke(f"Tell me about {topic}.")
print(f"  name       : {p.name}")
print(f"  description: {p.description[:200]}")
print(f"  score      : {p.score}")

In [ ]:
# ─── 4-B: The full EntityProfile schema from the companion script ──────────────
# This is the schema used in the workflow later in this notebook.
# Study the descriptions — each one constrains the model's output.

class EntityProfile(BaseModel):
    name: str = Field(description="Full name of the entity exactly as commonly written")
    summary: str = Field(
        description="2-3 sentence overview: what it is, who made it, why it matters"
    )
    key_facts: list[str] = Field(
        description="3-5 notable facts. Each fact is a single sentence. "
                    "Prefer specific, verifiable claims over general statements."
    )
    confidence: str = Field(
        description="Source quality estimate: 'high' if multiple reliable sources agree, "
                    "'medium' if one good source, 'low' if speculative or conflicting"
    )


print("EntityProfile JSON Schema:")
print(json.dumps(EntityProfile.model_json_schema(), indent=2))

---

## Part 5 — The LangGraph Pipeline: Search Then Extract

---

### Graph Architecture

```
  USER INPUT
  subject = "Anthropic"
       |
       v
  +--------------------------------------------------+
  |              LangGraph StateGraph                |
  |                                                  |
  |  START                                           |
  |    |                                             |
  |    v                                             |
  |  +------------------+                            |
  |  |   search_node    |  DuckDuckGo web search     |
  |  |                  |  max_results=5             |
  |  |  state["raw"]    |  -> raw text snippets      |
  |  +--------+---------+                            |
  |           |  raw_results (string)                |
  |           v                                      |
  |  +------------------+                            |
  |  |  extract_node    |  ChatOpenAI gpt-4o-mini    |
  |  |                  |  .with_structured_output() |
  |  |  state["prof"]   |  -> validated EntityProfile|
  |  +--------+---------+                            |
  |           |  profile (dict)                      |
  |          END                                     |
  +--------------------------------------------------+
       |
       v
  {"name": "Anthropic", "summary": "...",
   "key_facts": [...], "confidence": "high"}
```

### Why Two Nodes?

Separating **retrieval** from **extraction** is a fundamental agentic design principle:
- `search_node` is **stateless and swappable** — replace DuckDuckGo with any search API without touching extraction logic
- `extract_node` is **schema-focused** — it only cares about `raw_results`, not where they came from
- Each node can be tested, retried, and monitored independently

### State Schema

```python
class ProfileState(TypedDict):
    subject:     str   # Input — what to search for
    raw_results: str   # Intermediate — raw text from search
    profile:     dict  # Output — validated profile as dict
```

Each node returns a dict with the keys it updates. LangGraph merges that dict into the running state automatically.

In [ ]:
# ─── 5-A: Define state and search node ───────────────────────────────────────


class ProfileState(TypedDict):
    subject: str
    raw_results: str
    profile: dict


def search_web(query: str, max_results: int = 5) -> str:
    """Fetch DuckDuckGo snippets for a query and return as a single string."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
        if not results:
            return "No search results found."
        return "\n".join(f"{r['title']}: {r['body']}" for r in results)
    except Exception as e:
        return f"Search failed: {e}"


def search_node(state: ProfileState) -> dict:
    raw = search_web(state["subject"])
    print(f"[search_node] Retrieved {len(raw)} chars for '{state['subject']}'")
    return {"raw_results": raw}


# Quick test
test_raw = search_web("LangChain", max_results=2)
print(f"Search test: {len(test_raw)} chars")
print(test_raw[:300])

In [ ]:
# ─── 5-B: Define extract node and build the graph ─────────────────────────────

extract_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
structured_extract_llm = extract_llm.with_structured_output(EntityProfile)


def extract_node(state: ProfileState) -> dict:
    profile = structured_extract_llm.invoke(
        [
            SystemMessage(
                f"Extract a structured profile for: {state['subject']}. "
                f"Use only the information in the search results provided."
            ),
            HumanMessage(f"Search results:\n{state['raw_results']}"),
        ]
    )
    print(f"[extract_node] Extracted profile with confidence='{profile.confidence}'")
    return {"profile": profile.model_dump()}


def create_workflow():
    graph = StateGraph(ProfileState)
    graph.add_node("search", search_node)
    graph.add_node("extract", extract_node)
    graph.add_edge(START, "search")
    graph.add_edge("search", "extract")
    graph.add_edge("extract", END)
    return graph.compile()


app = create_workflow()
print("Graph compiled: START -> search -> extract -> END")

In [ ]:
# ─── 5-C: Run the pipeline on multiple subjects ────────────────────────────────

SUBJECTS = ["LangChain", "Andrej Karpathy"]

for subject in SUBJECTS:
    print(f"\nProfiling: {subject}")
    print("-" * 50)
    result = app.invoke({"subject": subject, "raw_results": "", "profile": {}})
    print(json.dumps(result["profile"], indent=2))

### Exercise 2 — Profile Your Own Subject

Run the pipeline on a subject of your choice — a company, a person, a technology, or an open-source project.

**Questions to consider:**
1. Does the `confidence` field reflect the quality of available search results?
2. Are the `key_facts` specific (with numbers, dates) or generic?
3. What happens when you profile something very recent or obscure?

In [ ]:
# ── Exercise 2 Starter ────────────────────────────────────────────────────────
MY_SUBJECT = "TODO: replace with your subject"

if not MY_SUBJECT.startswith("TODO"):
    result = app.invoke({"subject": MY_SUBJECT, "raw_results": "", "profile": {}})
    print(json.dumps(result["profile"], indent=2))
else:
    print("Replace MY_SUBJECT with a real value and re-run.")

---

## Part 6 — Schema Variants: Nested Models, Literals, Optional Fields

---

### Going Beyond Flat Schemas

Real-world extraction tasks often require:
- **Nested objects** — a `Company` with an embedded `Address`
- **Constrained choices** — a field that must be one of a fixed set of values
- **Optional fields** — data that may not be present in all sources

Pydantic handles all of these natively. LangChain passes the full nested schema to the model.

---

### Useful Type Annotations

| Annotation | Example | Meaning |
|------------|---------|----------|
| `Literal["a", "b", "c"]` | `status: Literal["active", "defunct", "unknown"]` | Enum-like constraint — only these values allowed |
| `Optional[T]` | `founded: Optional[int] = None` | Field may be null/missing |
| `list[T]` | `tags: list[str]` | Array of items |
| `NestedModel` | `address: Address` | Embedded sub-object |
| `list[NestedModel]` | `founders: list[Person]` | Array of sub-objects |

In [ ]:
# ─── 6-A: Literal types — constrained vocabulary ──────────────────────────────
# The model MUST return one of the specified strings.

class EntityProfileStrict(BaseModel):
    name: str = Field(description="Full name of the entity")
    summary: str = Field(description="2-3 sentence overview")
    key_facts: list[str] = Field(description="3-5 notable facts")
    # Literal enforces exactly one of these values — no free-form text
    confidence: Literal["high", "medium", "low"] = Field(
        description="'high' if multiple reliable sources agree, "
                    "'medium' if one good source, 'low' if speculative"
    )
    entity_type: Literal["person", "company", "technology", "concept", "other"] = Field(
        description="Category of the entity being profiled"
    )


strict_llm = llm.with_structured_output(EntityProfileStrict)

for subject in ["Guido van Rossum", "Stripe", "Docker"]:
    r = strict_llm.invoke(
        [
            SystemMessage(f"Extract a structured profile for: {subject}"),
            HumanMessage("Use your training knowledge."),
        ]
    )
    print(f"{r.name:30}  type={r.entity_type:12}  confidence={r.confidence}")

In [ ]:
# ─── 6-B: Optional fields and nested models ───────────────────────────────────

class Founder(BaseModel):
    name: str = Field(description="Full name of the founder")
    role: str = Field(description="Their role at founding, e.g. 'CEO', 'CTO', 'Co-founder'")


class CompanyProfile(BaseModel):
    name: str = Field(description="Official company name")
    founded_year: Optional[int] = Field(
        default=None,
        description="Year the company was founded. Null if unknown."
    )
    headquarters: Optional[str] = Field(
        default=None,
        description="City and country of HQ. Null if unknown."
    )
    founders: list[Founder] = Field(
        description="List of founders. Empty list if unknown."
    )
    industry: str = Field(description="Primary industry sector")
    summary: str = Field(description="2-3 sentence company overview")


company_llm = llm.with_structured_output(CompanyProfile)

company = company_llm.invoke(
    [
        SystemMessage("Extract a structured company profile."),
        HumanMessage("Tell me about Anthropic."),
    ]
)

print(f"Company  : {company.name}")
print(f"Founded  : {company.founded_year}")
print(f"HQ       : {company.headquarters}")
print(f"Industry : {company.industry}")
print(f"Founders :")
for f in company.founders:
    print(f"  - {f.name} ({f.role})")
print(f"Summary  : {company.summary[:150]}")
print()
print("Nested structure as dict:")
print(json.dumps(company.model_dump(), indent=2))

In [ ]:
# ─── 6-C: Add an optional founded_year field to EntityProfile ─────────────────
# Works for entities that may or may not have a founding date.

class EntityProfileV2(BaseModel):
    name: str = Field(description="Full name of the entity exactly as commonly written")
    summary: str = Field(description="2-3 sentence overview")
    key_facts: list[str] = Field(description="3-5 notable facts")
    confidence: Literal["high", "medium", "low"] = Field(
        description="Source quality estimate"
    )
    # Optional: null for people and abstract concepts
    founded_year: Optional[int] = Field(
        default=None,
        description="Year founded or established. Null for people or abstract concepts."
    )


v2_llm = llm.with_structured_output(EntityProfileV2)

for subject in ["Anthropic", "Sam Altman", "Retrieval Augmented Generation"]:
    r = v2_llm.invoke(
        [
            SystemMessage(f"Profile this entity: {subject}"),
            HumanMessage("Use your training knowledge."),
        ]
    )
    founded = str(r.founded_year) if r.founded_year else "N/A"
    print(f"{r.name:40}  founded={founded:6}  confidence={r.confidence}")

### Exercise 3 — Enforce a Literal Type

Change the `confidence` field in `EntityProfile` from `str` to `Literal["high", "medium", "low"]`.

**Then test it:**
1. Profile `"LangGraph"` — does the model comply?
2. Try setting `temperature=0.9` — does the Literal constraint still hold?

**Expected observation:** The Literal constraint is enforced at the API level, not the prompt level — temperature has no effect on whether the output value is schema-valid.

In [ ]:
# ── Exercise 3 Starter ────────────────────────────────────────────────────────

class EntityProfileLiteral(BaseModel):
    name: str = Field(description="Full name of the entity")
    summary: str = Field(description="2-3 sentence overview")
    key_facts: list[str] = Field(description="3-5 notable facts")
    # TODO: change confidence from str to Literal["high", "medium", "low"]
    confidence: str = Field(description="high | medium | low based on source quality")


# TODO: bind to llm.with_structured_output and profile "LangGraph"
# TODO: try temperature=0.9 and verify the Literal still holds

---

## Part 7 — Error Handling and Retry Strategies

---

### When Structured Output Fails

Even with `with_structured_output()`, failures can happen:

| Failure mode | Cause | Fix |
|-------------|-------|-----|
| `ValidationError` | Model returns data that fails Pydantic validation | Add `include_raw=True`, inspect `parsing_error` |
| Empty list for a required field | Model returns `[]` when description says 3-5 items | Strengthen field description |
| `None` for a non-optional field | Model returns null despite field being required | Make field `Optional` or tighten description |
| Rate limit error | Too many API calls | Add exponential backoff via `.with_retry()` |
| Search returns no results | DuckDuckGo rate limit or offline | Handle in `search_node` with fallback text |

---

### Defensive Pattern: Wrap with `include_raw=True`

```python
safe_llm = llm.with_structured_output(MyModel, include_raw=True)
result = safe_llm.invoke(messages)

if result["parsing_error"]:
    # log result["raw"] for debugging
    raise RuntimeError(f"Extraction failed: {result['parsing_error']}")

profile = result["parsed"]
```

In [ ]:
# ─── 7-A: Robust extraction with error handling ────────────────────────────────

safe_llm = llm.with_structured_output(EntityProfile, include_raw=True)


def safe_extract(subject: str, raw_text: str) -> EntityProfile | None:
    """
    Extract an EntityProfile with graceful error handling.
    Returns None if extraction fails.
    """
    try:
        result = safe_llm.invoke(
            [
                SystemMessage(f"Extract a structured profile for: {subject}"),
                HumanMessage(f"Use these search results:\n{raw_text}"),
            ]
        )
        if result["parsing_error"]:
            print(f"[warn] Parsing error for '{subject}': {result['parsing_error']}")
            return None
        return result["parsed"]
    except Exception as e:
        print(f"[error] Extraction failed for '{subject}': {e}")
        return None


# Test with real content
raw = search_web("LangGraph", max_results=3)
profile = safe_extract("LangGraph", raw)

if profile:
    print(f"Extraction succeeded: {profile.name}")
    print(f"Confidence: {profile.confidence}")
    print(f"Facts: {len(profile.key_facts)} items")
else:
    print("Extraction failed — check warnings above.")

In [ ]:
# ─── 7-B: .with_retry() for transient failures ────────────────────────────────
# LangChain runnables support .with_retry() for automatic retries on exceptions.
# This is especially useful for rate limit errors from the API.

resilient_llm = (
    llm.with_structured_output(EntityProfile)
       .with_retry(
           retry_if_exception_type=(Exception,),
           wait_exponential_jitter=True,
           stop_after_attempt=3,
       )
)

print("Resilient LLM configured with up to 3 retries on any exception.")
print("Retry runnable type:", type(resilient_llm).__name__)

# Use it exactly like the standard structured LLM
r = resilient_llm.invoke(
    [
        SystemMessage("Profile this entity."),
        HumanMessage("Tell me about NumPy."),
    ]
)
print(f"\nExtracted: {r.name} — {r.summary[:100]}")

---

## Part 8 * — Advanced Techniques (Bonus)

---

### Multi-Schema Routing

Some pipelines need to extract *different* schemas depending on the entity type. For example, profiling a person vs. a company requires different fields. You can implement this with a lightweight routing step:

```
subject -> classify_node -> person_schema  OR  company_schema -> extract_node
```

The classify node uses a `Literal` schema to pick the route; the extract node dispatches to the correct Pydantic model.

---

### Streaming Structured Output

For long extractions, you can stream partial results:

```python
for chunk in structured_llm.stream(messages):
    print(chunk)  # prints as fields are filled in
```

Note: partial Pydantic objects may not validate until the stream is complete.

---

### Using TypedDict Instead of Pydantic

For simpler cases where you do not need runtime validation, `TypedDict` is supported:

```python
from typing import TypedDict

class SimpleProfile(TypedDict):
    name: str
    summary: str

result: SimpleProfile = llm.with_structured_output(SimpleProfile).invoke(...)
# result is a plain dict — no Pydantic validation
```

In [ ]:
# ─── 8-A: Entity type routing ─────────────────────────────────────────────────

class EntityType(BaseModel):
    entity_type: Literal["person", "company", "technology", "other"] = Field(
        description="Classify the subject: person (human individual), "
                    "company (organization or business), "
                    "technology (software, hardware, protocol), or other"
    )


class PersonProfile(BaseModel):
    name: str = Field(description="Full name")
    born: Optional[int] = Field(default=None, description="Birth year, or null")
    nationality: Optional[str] = Field(default=None, description="Country of origin")
    known_for: list[str] = Field(description="2-4 things this person is known for")
    summary: str = Field(description="2-3 sentence biography")


classifier = llm.with_structured_output(EntityType)
person_extractor = llm.with_structured_output(PersonProfile)
company_extractor = llm.with_structured_output(CompanyProfile)
generic_extractor = llm.with_structured_output(EntityProfile)


def smart_profile(subject: str) -> dict:
    """Route to the right schema based on entity type."""
    entity = classifier.invoke(f"What type of entity is: {subject}?")
    msgs = [
        SystemMessage(f"Extract a detailed profile for: {subject}"),
        HumanMessage("Use your training knowledge."),
    ]
    if entity.entity_type == "person":
        result = person_extractor.invoke(msgs)
    elif entity.entity_type == "company":
        result = company_extractor.invoke(msgs)
    else:
        result = generic_extractor.invoke(msgs)
    return {"type": entity.entity_type, "data": result.model_dump()}


for subject in ["Yann LeCun", "Mistral AI", "LangGraph"]:
    out = smart_profile(subject)
    print(f"{subject:20}  type={out['type']:12}  name={out['data'].get('name')}")

---

## What's Next?

You now understand the full structured output pattern — from schema design through LangGraph pipelines with error handling. Here is where to go deeper:

### Apply structured output to evaluation
- **Example 29 — LLM Judge Harness** ([`../29-llm-judge-harness/`](../29-llm-judge-harness/)): structured output for rubric scoring — a `JudgeScore(relevance: int, faithfulness: int, completeness: int)` model scores RAG answers automatically. The pattern is identical to what you built here.

### Apply structured output to routing decisions
- **Example 25 — Adaptive RAG** ([`../25-adaptive-rag/`](../25-adaptive-rag/)): a `RouteDecision(strategy: Literal["vector", "keyword", "hybrid"])` model picks the retrieval strategy per query. One `with_structured_output()` call replaces an entire if-else routing chain.
- **Example 27 — Self-RAG** ([`../27-self-rag/`](../27-self-rag/)): `BoolDecision(answer: bool)` powers three LLM-as-judge gates (is the retrieval relevant? is the answer grounded? is it useful?).

### Go deeper on schema design
- **Pydantic validators** — use `@field_validator` for cross-field constraints and custom normalization
- **Discriminated unions** — `Union[PersonProfile, CompanyProfile]` with a `type` discriminator field
- **JSON Schema `$defs`** — Pydantic auto-generates references for nested models; inspect with `model_json_schema()`

### Production considerations
- Cache extractions — the same subject extracted twice wastes tokens; cache by subject hash
- Log `parsing_error` cases — they reveal schema-model mismatches early
- Version your schemas — changing a field name breaks downstream consumers

### Further reading
- OpenAI Structured Outputs: https://platform.openai.com/docs/guides/structured-outputs
- LangChain structured output concepts: https://python.langchain.com/docs/concepts/structured_outputs/
- Pydantic model validators: https://docs.pydantic.dev/latest/concepts/validators/
- Brown et al. (2020). *Language Models are Few-Shot Learners.* https://arxiv.org/abs/2005.14165

---

*Workshop complete. The next step is example 29 — apply structured output to automatic answer scoring.*

---

## Exercise Answer Key

Use this section after attempting the exercises yourself. These are sample solutions — not the only correct answers.

---

### Exercise 1 — Design a Schema

**What good descriptions look like:**
- `title`: "The full title of the book exactly as printed on the cover"
- `author`: "Full name of the primary author (not editor or contributor)"
- `published_year`: "Year of first publication as a 4-digit integer"
- `genres`: "1-3 genre labels, e.g. ['science fiction', 'thriller']. Use standard library classification terms."
- `one_line_summary`: "Exactly one sentence (ending in a period) summarizing the book's plot or subject matter"

**Why these are better:** Each description tells the model the *format* and *constraints*, not just the name. The genre description gives examples. The summary description specifies the sentence count and punctuation.

### Exercise 2 — Profile Your Own Subject

**What to look for:**
- `confidence='low'` is expected for very recent events or obscure topics — DuckDuckGo may return thin results
- `key_facts` should contain specific numbers, dates, or names rather than generic statements like "it is widely used"
- If the subject is very new (within a few months), the LLM's training knowledge may dominate over search results

**A strong result** has `key_facts` with at least one verifiable claim (a date, a funding amount, a user count) and a `confidence` that matches how well-sourced the subject actually is.

### Exercise 3 — Enforce a Literal Type

**The key insight:** When you change `confidence: str` to `confidence: Literal["high", "medium", "low"]`, Pydantic adds an `"enum"` constraint to the JSON Schema. The OpenAI API enforces this at token generation — the model is literally unable to output any other value for that field. Temperature affects the *distribution* over valid tokens, not whether the output is schema-compliant.

In [ ]:
# Answer key — Exercise 1
class BookProfileAnswer(BaseModel):
    title: str = Field(
        description="The full title of the book exactly as printed on the cover"
    )
    author: str = Field(
        description="Full name of the primary author (not editor or contributor)"
    )
    published_year: int = Field(
        description="Year of first publication as a 4-digit integer"
    )
    genres: list[str] = Field(
        description="1-3 genre labels, e.g. ['science fiction', 'thriller']. "
                    "Use standard library classification terms."
    )
    one_line_summary: str = Field(
        description="Exactly one sentence (ending in a period) summarizing "
                    "the book's plot or subject matter"
    )


book_llm = llm.with_structured_output(BookProfileAnswer)
book = book_llm.invoke(
    "Tell me about 'The Pragmatic Programmer' by Andrew Hunt and David Thomas."
)
print(f"Title       : {book.title}")
print(f"Author      : {book.author}")
print(f"Published   : {book.published_year}")
print(f"Genres      : {book.genres}")
print(f"Summary     : {book.one_line_summary}")

In [ ]:
# Answer key — Exercise 3
class EntityProfileLiteralAnswer(BaseModel):
    name: str = Field(description="Full name of the entity")
    summary: str = Field(description="2-3 sentence overview")
    key_facts: list[str] = Field(description="3-5 notable facts")
    # Changed: str -> Literal["high", "medium", "low"]
    confidence: Literal["high", "medium", "low"] = Field(
        description="'high' if multiple reliable sources agree, "
                    "'medium' if one good source, 'low' if speculative"
    )


# Verify the enum appears in the JSON Schema
schema = EntityProfileLiteralAnswer.model_json_schema()
confidence_schema = schema["properties"]["confidence"]
print("confidence field schema:", json.dumps(confidence_schema, indent=2))
print()

# Test with temperature=0.9 — Literal still enforced at the API level
high_temp_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.9)
literal_llm = high_temp_llm.with_structured_output(EntityProfileLiteralAnswer)

r = literal_llm.invoke(
    [
        SystemMessage("Profile this entity."),
        HumanMessage("Tell me about LangGraph."),
    ]
)
print(f"confidence value : {r.confidence!r}")
print(f"Valid Literal    : {r.confidence in ('high', 'medium', 'low')}")